In [14]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

def get_investment_recommendations(investor_id, csv_investors='dummy_investors.csv', csv_opportunities='dummy_opportunities.csv', top_k=5):
  df_investors = pd.read_csv(csv_investors)
  df_opportunities = pd.read_csv(csv_opportunities)

  location = [
      "أبها", "خميس مشيط", "بيشة", "محايل عسير", "تثليث",
      "النماص", "ظهران الجنوب", "سراة عبيدة", "رجال ألمع",
      "بلقرن", "أحد رفيدة", "تنومة", "بارق", "المجاردة",
      "طريب", "البرك", "الحرجة", "الأمواه"
  ]

  category = [
      "مقاهي ومشروبات",
      "مطاعم وأغذية",
      "زراعة ومنتجات طبيعية",
      "حرف يدوية وتراثية",
      "سياحة وضيافة",
      "منتجات عسلية",
      "صناعات غذائية محلية",
      "خدمات نقل وتوصيل"
  ]

  budget = [
      "أقل من 5,000 ريال",
      "5,000 - 15,000 ريال",
      "15,000 - 50,000 ريال",
      "50,000 - 100,000 ريال",
      "100,000 - 300,000 ريال",
      "أكثر من 300,000 ريال"
  ]

  expected_roi = [
      "5% - 10%", "10% - 15%", "15% - 25%",
      "25% - 40%", "40% - 60%", "أكثر من 60%"
  ]

  risk_levels = ["منخفض", "متوسط", "مرتفع"]

  # map
  location_mapping = {location[i]: i+1 for i in range(len(location))}
  category_mapping = {category[i]: i+1 for i in range(len(category))}
  budget_mapping = {budget[i]: i+1 for i in range(len(budget))}
  expected_roi_mapping = {expected_roi[i]: i+1 for i in range(len(expected_roi))}
  risk_levels_mapping = {risk_levels[i]: i+1 for i in range(len(risk_levels))}

  # check if the provided investor id's row exists
  investor = df_investors[df_investors['nationalID'] == investor_id].iloc[0]
  investor_vec = np.array([budget_mapping[investor['preffered_budget']], category_mapping[investor['preffered_fieldOfInterest']], location_mapping[investor['preffered_geographicalRegion']], risk_levels_mapping[investor['preffered_risks']], expected_roi_mapping[investor['preffered_expectedROI']]]).reshape(1, -1)

  # look through active opps
  active_opps = df_opportunities[df_opportunities['opportunityStatus'] == 'نشط']

  similarities = []

  # loop through and compute the similarity between the user and each opportunity
  for idx, opp in active_opps.iterrows():
      opp_vec = np.array([budget_mapping[opp['requiredInvestmentBudget']], category_mapping[opp['category']], location_mapping[opp['location']], risk_levels_mapping[opp['riskLevel']], expected_roi_mapping[opp['expectedROI']]]).reshape(1, -1)
      similarity = cosine_similarity(investor_vec, opp_vec)[0][0]
      similarities.append({'opportunity_id': opp['opportunityID'], 'name': opp['name'], 'category': opp['category'], 'location': opp['location'], 'budget': opp['requiredInvestmentBudget'], 'risk': opp['riskLevel'], 'roi': opp['expectedROI'], 'similarity_score': similarity})

  # sort from the most matching/suitable opp to the least
  similarities.sort(key=lambda x: x['similarity_score'], reverse=True)

  return similarities[:top_k]

In [15]:
investorid = 1012412345

In [16]:
get_investment_recommendations(investor_id=investorid)

[{'opportunity_id': 1,
  'name': 'قطرات الجنوب',
  'category': 'مقاهي ومشروبات',
  'location': 'أبها',
  'budget': '50,000 - 100,000 ريال',
  'risk': 'منخفض',
  'roi': '15% - 25%',
  'similarity_score': np.float64(0.9923721235559482)},
 {'opportunity_id': 21,
  'name': 'متجر ثمار الجنوب',
  'category': 'زراعة ومنتجات طبيعية',
  'location': 'بيشة',
  'budget': '15,000 - 50,000 ريال',
  'risk': 'متوسط',
  'roi': '10% - 15%',
  'similarity_score': np.float64(0.8136375599905377)},
 {'opportunity_id': 7,
  'name': 'نقاء الجنوب',
  'category': 'صناعات غذائية محلية',
  'location': 'محايل عسير',
  'budget': '100,000 - 300,000 ريال',
  'risk': 'مرتفع',
  'roi': '25% - 40%',
  'similarity_score': np.float64(0.7889148743691712)},
 {'opportunity_id': 29,
  'name': 'عربة قهوة متنقلة',
  'category': 'مقاهي ومشروبات',
  'location': 'خميس مشيط',
  'budget': 'أقل من 5,000 ريال',
  'risk': 'مرتفع',
  'roi': '10% - 15%',
  'similarity_score': np.float64(0.73620192955128)},
 {'opportunity_id': 23,
  'name